In [7]:
import pandas as pd
import mysql.connector

# Read CSV
df = pd.read_csv("attribution_ready.csv")

# Connect MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="mysql",
    database="marketing_attribution"
)

cursor = conn.cursor()

# Insert rows
query = """
INSERT INTO marketing_data
(user_id,timestamp,channel,campaign,conversion_flag,touchpoint_order)
VALUES (%s,%s,%s,%s,%s,%s)
"""

for _, row in df.iterrows():
    cursor.execute(query, (
        int(row["user_id"]),
        str(row["timestamp"]),
        str(row["channel"]),
        str(row["campaign"]),
        int(row["conversion_flag"]),
        int(row["touchpoint_order"])
    ))

conn.commit()

print("Data Loaded Successfully")

Data Loaded Successfully


In [9]:
df.shape

(10000, 6)

In [13]:
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="mysql",
    database="marketing_attribution"
)

query = """
WITH first_touch AS
(
    SELECT
        user_id,
        channel,
        ROW_NUMBER() OVER(
            PARTITION BY user_id
            ORDER BY timestamp
        ) rn
    FROM marketing_data
)
SELECT
    channel,
    COUNT(*) first_touch_conversions
FROM first_touch
WHERE rn=1
GROUP BY channel
"""

df = pd.read_sql(query, conn)

df.to_csv(
    "first_touch_summary.csv",
    index=False
)

C:\Users\Dell\AppData\Local\Temp\ipykernel_3472\1523821864.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [15]:
import mysql.connector

try:
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="mysql",
        database="marketing_attribution"
    )

    print("Connected Successfully")

except Exception as e:
    print(e)

Connected Successfully


In [17]:
import pandas as pd

query = """
WITH first_touch AS
(
    SELECT
        user_id,
        channel,
        ROW_NUMBER() OVER(
            PARTITION BY user_id
            ORDER BY timestamp
        ) AS rn
    FROM marketing_data
)

SELECT
    channel,
    COUNT(*) AS first_touch_conversions
FROM first_touch
WHERE rn = 1
GROUP BY channel
ORDER BY first_touch_conversions DESC
"""

df_first = pd.read_sql(query, conn)

df_first.to_csv(
    "first_touch_summary.csv",
    index=False
)

print(df_first.head())

          channel  first_touch_conversions
0     Display Ads                      492
1  Direct Traffic                      489
2        Referral                      486
3    Social Media                      469
4      Search Ads                      461


C:\Users\Dell\AppData\Local\Temp\ipykernel_3472\4275906302.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_first = pd.read_sql(query, conn)


In [19]:
query = """
WITH last_touch AS
(
    SELECT
        user_id,
        channel,
        ROW_NUMBER() OVER(
            PARTITION BY user_id
            ORDER BY timestamp DESC
        ) AS rn
    FROM marketing_data
)

SELECT
    channel,
    COUNT(*) AS last_touch_conversions
FROM last_touch
WHERE rn = 1
GROUP BY channel
ORDER BY last_touch_conversions DESC
"""

df_last = pd.read_sql(query, conn)

df_last.to_csv(
    "last_touch_summary.csv",
    index=False
)

C:\Users\Dell\AppData\Local\Temp\ipykernel_3472\3009445424.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_last = pd.read_sql(query, conn)


In [23]:
print(df_last.head())

          channel  last_touch_conversions
0  Direct Traffic                     511
1      Search Ads                     480
2           Email                     475
3     Display Ads                     468
4    Social Media                     458


In [25]:
pd.read_sql(query, conn)

C:\Users\Dell\AppData\Local\Temp\ipykernel_3472\3931869133.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query, conn)


,channel,last_touch_conversions
0,Direct Traffic,511
1,Search Ads,480
2,Email,475
3,Display Ads,468
4,Social Media,458
5,Referral,455


In [27]:
cursor = conn.cursor()

cursor.execute(query)

rows = cursor.fetchall()

for row in rows[:10]:
    print(row)

('Direct Traffic', 511)
('Search Ads', 480)
('Email', 475)
('Display Ads', 468)
('Social Media', 458)
('Referral', 455)


In [33]:
query = """
SELECT
    channel,
    ROUND(SUM(attribution_weight),2) AS linear_credit
FROM
(
    SELECT
        channel,
        1.0 /
        COUNT(*) OVER(PARTITION BY user_id)
        AS attribution_weight
    FROM marketing_data
) t
GROUP BY channel
ORDER BY linear_credit DESC
"""

cursor = conn.cursor()
cursor.execute(query)

rows = cursor.fetchall()

columns = [i[0] for i in cursor.description]

df_linear = pd.DataFrame(rows, columns=columns)

df_linear.to_csv(
    "linear_attribution_summary.csv",
    index=False
)

print(df_linear)

          channel linear_credit
0  Direct Traffic        487.81
1    Social Media        475.08
2        Referral        473.41
3     Display Ads        471.91
4      Search Ads        470.58
5           Email        468.22


In [35]:
!git status

fatal: not a git repository (or any of the parent directories): .git
